# Social Media Usage Survey — Data Inspection & Preprocessing

This notebook loads the survey data, standardizes the column names, checks data quality, produces descriptive statistics and visualizations, and saves a cleaned dataset.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)


## 1. Load the survey data

If the original Excel file is present, it is used. Otherwise the reconstructed CSV included with this repository is used.

In [ ]:
EXCEL_FILE = "responses (survey).xlsx"
CSV_FILE = "data/responses_survey_reconstructed.csv"

if os.path.exists(EXCEL_FILE):
    df = pd.read_excel(EXCEL_FILE)
    print(f"Loaded Excel file: {EXCEL_FILE}")
elif os.path.exists(CSV_FILE):
    df = pd.read_csv(CSV_FILE)
    print(f"Loaded reconstructed CSV: {CSV_FILE}")
else:
    raise FileNotFoundError("Place responses (survey).xlsx in the project folder, or keep the included CSV in data/.")

print("Shape:", df.shape)
df.head()


## 2. Standardize column names

In [ ]:
COLUMN_MAP = {
    "Timestamp": "timestamp",
    "What is your gender?": "gender",
    "What is your age?": "age",
    "What is your current academic year?": "academic_year",
    "What is your field of study?": "field",
    "Which social media platform do you use most frequently?": "platform",
    "What is your primary purpose for using social media?": "purpose",
    "Approximately how many hours per day do you spend on social media?": "sm_hours",
    "How many different social media platforms do you actively use?": "active_platforms",
    "Approximately how many hours per day do you spend studying or doing academic work outside of classes?": "study_hours",
    "On average , how many hours do you sleep per night?": "sleep_hours",
    "Approximately how many hours do you spend using your phone within the 1 hour before going to sleep?": "phone_last_hour",
    "Do you usually use social media within 30 minutes before going to sleep?": "social_before_sleep",
    "How often does social media distract you while studying?": "study_distraction",
    "Have you ever intentionally tried to reduce your social media usage?": "tried_reduce",
    "How difficult would it be for you to stay away from social media for one full day?": "difficulty_away",
}

NUMERIC_COLS = ["age", "sm_hours", "active_platforms", "study_hours", "sleep_hours", "phone_last_hour", "difficulty_away"]

# If the source is the original Google Forms Excel export, rename its long questions.
df = df.rename(columns=COLUMN_MAP)

# Normalize text fields and numeric fields.
for col in ["gender", "academic_year", "field", "platform", "purpose", "social_before_sleep", "study_distraction", "tried_reduce"]:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

for col in NUMERIC_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print(df.columns.tolist())
df.head()


## 3. Data-quality checks

In [ ]:
print("Missing values:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nData types:")
print(df.dtypes)


## 4. Frequency tables for categorical variables

In [ ]:
categorical_cols = ["gender", "academic_year", "field", "platform", "purpose", "social_before_sleep", "study_distraction", "tried_reduce"]
for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


## 5. Descriptive statistics

In [ ]:
numeric_cols = ["age", "sm_hours", "active_platforms", "study_hours", "sleep_hours", "phone_last_hour", "difficulty_away"]
desc = df[numeric_cols].describe().T
desc["IQR"] = desc["75%"] - desc["25%"]
desc["range"] = desc["max"] - desc["min"]
desc


## 6. Five-number summary and z-scores

In [ ]:
five_number = df[numeric_cols].agg(["min", "25%", "50%", "75%", "max"]).T
print(five_number)

z_scores = (df[numeric_cols] - df[numeric_cols].mean()) / df[numeric_cols].std(ddof=1)
print("\nFirst five rows of z-scores:")
z_scores.head()


## 7. Visualizations

In [ ]:
# Categorical bar chart: most frequently used platform
df["platform"].value_counts(dropna=False).plot(kind="bar")
plt.title("Most Frequently Used Social Media Platform")
plt.xlabel("Platform")
plt.ylabel("Number of respondents")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Histogram: daily social-media hours
df["sm_hours"].plot(kind="hist", bins=8, edgecolor="black")
plt.title("Daily Social Media Usage")
plt.xlabel("Hours per day")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# Box plot: daily social-media hours
df.boxplot(column="sm_hours")
plt.title("Box Plot of Daily Social Media Usage")
plt.ylabel("Hours per day")
plt.show()


## 8. Save cleaned data

In [ ]:
os.makedirs("data", exist_ok=True)
output_file = "data/cleaned_social_media_survey.csv"
df.to_csv(output_file, index=False)
print(f"Saved: {output_file}")
